# Real-Time Road Accident Hotspot Prediction and Visualization System
## MCA Final Year Major Project Demonstration Notebook
**Project Title:** RoadSafe AI  
**Author / Candidate:** Master of Computer Applications (MCA)  
**Technology Stack:** Python 3, Flask, MySQL / SQLite, Pandas, NumPy, Scikit-learn (Random Forest), GeoPandas, Folium, Joblib

---
### Notebook Objectives
This interactive notebook guides evaluators and students through the entire Machine Learning and Spatial Computing lifecycle:
1. Environment Setup & Dependency Installation
2. Dataset Loading / Synthetic Data Generation (6,000 Spatial Records)
3. Data Cleaning & Validation Pipeline
4. Exploratory Data Analysis (EDA) & Visualizations
5. Geospatial Analysis using GeoPandas (CRS EPSG:4326)
6. Feature Engineering & Categorical Transformation
7. Random Forest Classifier Training (120 Estimators, Class Balanced)
8. Comprehensive Model Evaluation (Accuracy, Precision, Recall, Confusion Matrix)
9. Persisting Trained Models via Joblib
10. Deploying and Launching the Live Flask Web Application


### Section 1: Installing Core Dependencies
Execute the following cell to install all required libraries from `requirements.txt`.


In [ ]:
# Step 1: Install required project packages
!pip install -r requirements.txt


### Section 2: Importing Libraries
Import the core packages for data manipulation, spatial GIS mapping, modeling, and metrics.


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("All dependencies successfully imported.")


### Section 3: Dataset Generation and Loading
Load the 6,000 spatial accident records. If not already generated, run `data/generate_data.py` to create the reproducible dataset.


In [ ]:
# Generate dataset if it does not already exist
data_path = Path("../data/accident_data.csv")
if not data_path.exists():
    print("Generating synthetic accident dataset...")
    !python ../data/generate_data.py

df = pd.read_csv(data_path)
print(f"Total Records Loaded: {len(df):,}")
print("Shape of Dataset:", df.shape)
df.head()


### Section 4: Data Cleaning & Coordinate Verification
Inspect missing values, verify latitude/longitude spatial bounds, and check class balance.


In [ ]:
# Data Cleaning & Validation
print("Missing Value Analysis:")
print(df.isnull().sum())

# Spatial coordinate boundary verification (Bengaluru / India Metropolitan bounds)
valid_coords = df[(df['latitude'] >= 12.0) & (df['latitude'] <= 14.0) &
                  (df['longitude'] >= 77.0) & (df['longitude'] <= 78.5)]
print(f"Valid Coordinates: {len(valid_coords)} / {len(df)}")

print("\nRisk Level Distribution:")
print(df['risk_level'].value_counts())


### Section 5: Exploratory Data Analysis (EDA)
Visualize accident distributions across hourly intervals, weather conditions, and risk levels.


In [ ]:
plt.figure(figsize=(14, 5))

# Plot 1: Accidents by Weather Condition
plt.subplot(1, 2, 1)
sns.countplot(data=df, x='weather', hue='risk_level', palette={'High': 'red', 'Medium': 'orange', 'Low': 'green'})
plt.title('Accident Severity Risk by Weather Condition')
plt.xlabel('Weather')
plt.ylabel('Incident Count')

# Plot 2: Traffic Volume Distribution by Risk Level
plt.subplot(1, 2, 2)
sns.boxplot(data=df, x='risk_level', y='traffic_volume', palette={'High': 'red', 'Medium': 'orange', 'Low': 'green'})
plt.title('Traffic Volume vs Accident Risk Level')
plt.xlabel('Risk Level')
plt.ylabel('Traffic Volume (vehicles/hr)')

plt.tight_layout()
plt.show()


### Section 6: Geographic Analysis using GeoPandas
Convert tabular coordinates into a formal GeoDataFrame using EPSG:4326 Coordinate Reference System and inspect spatial hotspots.


In [ ]:
# Construct GeoDataFrame
geometry = [Point(xy) for xy in zip(df['longitude'], df['latitude'])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

print("GeoDataFrame Information:")
print(f"CRS: {gdf.crs}")
print(f"Geometry type: {gdf.geometry.geom_type.iloc[0]}")

# Top 5 accident-dense locations
print("\nTop 5 Accident Hotspot Locations:")
print(gdf['location'].value_counts().head(5))


### Section 7: Feature Engineering & Preprocessing Pipeline
Extract temporal components (hour, day of week, month), encode categorical attributes with `OneHotEncoder`, and construct the feature matrix.


In [ ]:
sys.path.insert(0, str(Path("../").resolve()))
from ml.preprocessing import AccidentPreprocessor, RISK_CLASSES

preprocessor = AccidentPreprocessor()
preprocessor.fit(df)
X = preprocessor.transform(df)
y = df['risk_level'].values

print("Transformed Feature Matrix Shape:", X.shape)
print("Extracted Features Count:", len(preprocessor.feature_names))
print("Sample Features:", preprocessor.feature_names[:6])


### Section 8: Model Training with Random Forest
Partition data into 80% train and 20% test splits using stratified sampling, and train a 120-tree Random Forest Classifier.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training Samples: {X_train.shape[0]} | Testing Samples: {X_test.shape[0]}")

rf = RandomForestClassifier(
    n_estimators=120,
    max_depth=16,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)
rf.fit(X_train, y_train)
print("Random Forest Classifier trained successfully.")


### Section 9: Model Evaluation & Metrics
Evaluate classification performance, compute confusion matrix, and graph top feature importances.


In [ ]:
y_pred = rf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Test Set Accuracy: {acc * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=RISK_CLASSES))

# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred, labels=RISK_CLASSES)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=RISK_CLASSES, yticklabels=RISK_CLASSES)
plt.title('Multi-Class Confusion Matrix')
plt.xlabel('Predicted Risk')
plt.ylabel('Actual Risk')
plt.show()


### Section 10: Persisting Trained Model Artifacts
Persist the trained Random Forest model and preprocessor to disk using `joblib`.


In [ ]:
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

model_file = models_dir / "accident_hotspot_model.pkl"
prep_file = models_dir / "preprocessing.pkl"

joblib.dump(rf, model_file)
joblib.dump(preprocessor, prep_file)

print(f"Model saved to: {model_file}")
print(f"Preprocessor saved to: {prep_file}")


### Section 11: Launching the Flask Web Application
You can launch the complete Flask web application either directly using `%run ../app.py` or in a non-blocking background thread as demonstrated below.


In [ ]:
# Non-blocking Flask runner for Jupyter Notebook
import threading
from app import create_app

def run_flask_in_background():
    app = create_app()
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask_in_background, daemon=True)
flask_thread.start()

print("=" * 60)
print("RoadSafe AI Flask Application running in background thread!")
print("Open your browser and navigate to: http://127.0.0.1:5000")
print("=" * 60)
